# 09 Country Strategy Clustering

Unsupervised segmentation of countries by nuclear strategy using KMeans + PCA. Groups reveal structurally different market contexts for nuclear expansion.

> Run `python run_pipeline.py` first.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from joblib import load

countries = pd.read_csv(processed / 'country_nuclear_profile.csv')
clusters = pd.read_csv(predictions / 'country_clusters.csv')
metrics_path = ROOT / 'outputs' / 'metrics' / 'clustering_metrics.json'
import json
m = json.loads(metrics_path.read_text())
print(f"Clusters: {m['n_clusters']}  |  Silhouette score: {m['silhouette_score']:.3f}")
clusters.groupby('cluster_name').size().rename('countries')

## PCA scatter — country strategy segments

In [ ]:
merged = clusters.merge(countries[['country','region','operating_capacity_mwe','nuclear_share_percent','policy_signal_score']], on='country', how='left')

fig = px.scatter(
    merged,
    x='pca_x', y='pca_y',
    color='cluster_name', hover_name='country',
    hover_data=['region','operating_capacity_mwe','nuclear_share_percent','policy_signal_score'],
    size='operating_capacity_mwe'.split(),  # dummy for uniform size
    title='Country Nuclear Strategy Clusters (PCA projection)',
    template='plotly_white',
    labels={'pca_x': 'PC1', 'pca_y': 'PC2'}
)
fig.show()

## Cluster profiles — what makes each group different

In [ ]:
FEATURES = ['operating_capacity_mwe','construction_capacity_mwe','planned_capacity_mwe',
           'proposed_capacity_mwe','nuclear_share_percent','gdp_current_usd',
           'electricity_generation_twh','population','average_fleet_age',
           'advanced_reactor_activity_score','policy_signal_score']

profile = (
    clusters.merge(countries[['country'] + FEATURES], on='country', how='left')
    .groupby('cluster_name')[FEATURES]
    .mean()
    .round(1)
)
profile.T

In [ ]:
# Radar chart: normalized cluster profiles
from matplotlib.patches import FancyArrowPatch

plot_features = ['operating_capacity_mwe','nuclear_share_percent',
                 'average_fleet_age','policy_signal_score','advanced_reactor_activity_score']
labels = ['Operating
capacity','Nuclear
share','Fleet
age','Policy
signal','Advanced
reactor']

X_plot = profile[plot_features].copy()
X_norm = (X_plot - X_plot.min()) / (X_plot.max() - X_plot.min() + 1e-9)

angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
colors = ['#2196F3','#FF9800','#4CAF50']

for (name, row), color in zip(X_norm.iterrows(), colors):
    values = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, values, 'o-', linewidth=2, label=name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, size=10)
ax.set_title('Cluster Profiles (normalized)', pad=20, fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15))
plt.tight_layout(); plt.show()

## Which countries are in each cluster?

In [ ]:
for cluster, group in clusters.groupby('cluster_name'):
    print(f"\n{'='*50}")
    print(f"  {cluster}")
    print(f"{'='*50}")
    countries_in = group['country'].tolist()
    print(', '.join(countries_in))

## Silhouette analysis — how well-separated are the clusters?

In [ ]:
FEATURES = ['operating_capacity_mwe','construction_capacity_mwe','planned_capacity_mwe',
           'proposed_capacity_mwe','nuclear_share_percent','gdp_current_usd',
           'electricity_generation_twh','population','average_fleet_age',
           'advanced_reactor_activity_score','policy_signal_score']

X = countries[FEATURES].fillna(0)
X_scaled = StandardScaler().fit_transform(X)

scores = []
k_range = range(2, min(7, len(countries)-1))
for k in k_range:
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    s = silhouette_score(X_scaled, labels)
    scores.append({'k': k, 'silhouette': round(s, 3)})

df_scores = pd.DataFrame(scores)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_scores['k'], df_scores['silhouette'], 'o-', color='steelblue', linewidth=2)
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette Score vs Number of Clusters')
ax.set_xticks(list(k_range))
plt.tight_layout(); plt.show()
print(df_scores.to_string(index=False))

**Interpretation:** With sample data (16 countries), silhouette scores are low because the sample doesn't have enough diversity to form tight clusters. With the full IAEA + World Bank dataset (~50+ nuclear-relevant countries), clusters would be more meaningful and interpretable.

## What clustering tells us strategically

In [ ]:
print("""
Cluster use cases in a real advisory context:

1. Market screening: which cluster represents first-mover opportunities vs saturated markets?
2. Technology fit: do SMR vendors target entrant countries or aging-fleet replacers?
3. Policy benchmarking: how does a country's policy signal compare to its cluster peers?
4. Financing patterns: do high-income mature operators fund differently than emerging entrants?

These questions drive the cluster design choices: why fleet age, advanced-reactor activity score,
and policy signal are included alongside the economic indicators.
""")